In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("new_token")

In [2]:
import huggingface_hub
import os
import sys
import torch
import shutil
import json
import safetensors.torch

In [3]:
huggingface_hub.login(secret_value_0)

user = huggingface_hub.HfApi().whoami()['name']
print(user)

quark28


In [4]:
path_classic_directory = '/kaggle/working/hf/classic'
path_thinking_directory = '/kaggle/working/hf/thinking'
path_space_directory = '/kaggle/working/hf/space'
classic_model_path = '/kaggle/input/datasets/absolute1000/classic-model-after-sft/SFT_BEST.pt'
thinking_model_path = '/kaggle/input/datasets/absolute1000/model-after-sft/SFT_BEST.pt'

device = 'gpu' if torch.cuda.is_available() else 'cpu'

os.makedirs(path_classic_directory, exist_ok=True)
os.makedirs(path_classic_directory + '/onnx/', exist_ok=True)
os.makedirs(path_thinking_directory, exist_ok=True)
os.makedirs(path_space_directory, exist_ok=True)

In [5]:
classic_sft = {
    "PIQA": 54.5,
    "HellaSwag*": 26.7,
    "ARC-e*": 31.9,
    "ARC-c*": 21.0,
    "WinoG": 49.2,
    "SciQ": 48.7,
    "LAMBADA": 0.0,
    "BLiMP": 56.0
}

thinking_sft = {
    "PIQA": 52.6,
    "HellaSwag*": 26.0,
    "ARC-e*": 30.3,
    "ARC-c*": 22.4,
    "WinoG": 51.9,
    "SciQ": 33.7,
    "LAMBADA": 0.0,
    "BLiMP": 58.9
}

gpt2_124m = {
    "PIQA": 62.9,
    "HellaSwag*": 31.1,
    "ARC-e*": 39.5,
    "ARC-c*": 22.7,
    "WinoG": 51.6,
    "SciQ": 75.2,
    "LAMBADA": 32.6,
    "BLiMP": 82.1
}

pythia_14m = {
    "PIQA": 56.9,
    "HellaSwag*": 26.5,
    "ARC-e*": 32.7,
    "ARC-c*": 21.0,
    "WinoG": 51.3,
    "SciQ": 50.4,
    "LAMBADA": 10.3,
    "BLiMP": 67.7
}

In [6]:
%%writefile {path_classic_directory}/modeling_classic.py

import torch
import math

def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Classic(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

Writing /kaggle/working/hf/classic/modeling_classic.py


In [7]:
%%writefile {path_thinking_directory}/modeling_thinking.py

import torch
import math

def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class Thinking(torch.nn.Module):
    def __init__(self, emb_size, internal_state_size=128, vocab_size=12288, momentum=0.1):
        super().__init__()
        self.momentum = momentum
        self.to_vocab = torch.nn.Linear(emb_size, vocab_size, bias=False)
        self.alpha = torch.nn.Parameter(torch.ones(vocab_size))
        self.SiLU = torch.nn.SiLU()

        self.to_internal = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.ln = torch.nn.LayerNorm(internal_state_size)
        self.to_emb = torch.nn.Linear(internal_state_size, emb_size, bias=False)

        self.register_buffer("e_mean", torch.zeros(vocab_size))
        self.t_in = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.t_ln = torch.nn.LayerNorm(internal_state_size)
        self.t_out = torch.nn.Linear(internal_state_size, emb_size, bias=False)

    def forward(self, x, mask_for_thinking, context_ids):
        # x [B, Cs, Emb], mask_for_thinking [B, Cs, 1], context_ids [B, Cs]
        x_unit = x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        vocab_unit = self.to_vocab.weight / self.to_vocab.weight.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        energy = x_unit @ vocab_unit.T # [B, Cs, V], from -1 to 1

        own = torch.zeros_like(energy, dtype=torch.bool).scatter_(2, context_ids[..., None], True)
        drop = (~mask_for_thinking) | own # padding + the position's own token
        kept = energy.masked_fill(drop, 0.0) # [B, Cs, V]

        # words channel
        internal_state = kept * self.alpha
        internal_state = self.to_internal(internal_state)
        internal_state = self.ln(internal_state)
        internal_state = self.SiLU(internal_state)
        words = self.to_emb(internal_state) # [B, Cs, Emb]

        # thought channel
        e = kept.sum(dim=1).float() # [B, V]: sum of energies over positions
        if self.training:
            mu = e.mean(dim=0) # mean over texts in the batch
            with torch.no_grad():
                self.e_mean.mul_(1 - self.momentum).add_(self.momentum * mu)
        else:
            mu = self.e_mean # inference: running mean
        e = e - mu # how this text differs from the others
        e = e / e.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-6) # unit scale

        thought = self.t_in(e)
        thought = self.t_ln(thought)
        thought = self.SiLU(thought)
        thought = self.t_out(thought) # [B, Emb]

        thinking_out = words + thought.unsqueeze(1) # thought is added to every position
        return thinking_out

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Thinking(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Thinking = Thinking(embedding_size, thinking_internal_state_size)
        self.Thinking.to_vocab.weight = self.Embedding.weight
        self.out_norm = torch.nn.RMSNorm(embedding_size)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)
        torch.nn.init.zeros_(self.Thinking.t_out.weight)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        memory = self.Thinking(memory, mask.transpose(1, 2), context_ids)
        memory = self.out_norm(memory)
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

Writing /kaggle/working/hf/thinking/modeling_thinking.py


In [8]:
sys.path.insert(0, path_classic_directory)
sys.path.insert(0, path_thinking_directory)

In [9]:
from modeling_classic import NN_Classic
from modeling_thinking import NN_Thinking

In [10]:
config_classic = {
    "embedding_size": 384,
    "encoder_num": 6,
    "heads_encoder": 6,
    "decoder_num": 2,
    "heads_decoder": 6,
}

config_thinking = {
    "embedding_size": 384,
    "encoder_num": 4,
    "heads_encoder": 6,
    "thinking_internal_state_size": 128,
    "decoder_num": 2,
    "heads_decoder": 6,
}

In [11]:
model_classic = NN_Classic(**config_classic)
ckpt = torch.load(classic_model_path, map_location=device, weights_only=False)
model_classic.load_state_dict(ckpt['model'] )
model_classic.eval()
sum(p.numel() for p in model_classic.parameters())

20061696

In [12]:
model_thinking = NN_Thinking(**config_thinking)
ckpt = torch.load(thinking_model_path, map_location=device, weights_only=False)
model_thinking.load_state_dict(ckpt['model'] )
model_thinking.eval()
sum(p.numel() for p in model_thinking.parameters())

19778432

In [13]:
safetensors.torch.save_model(model_classic, path_classic_directory + '/model.safetensors')

In [14]:
safetensors.torch.save_model(model_thinking, path_thinking_directory + '/model.safetensors')

In [15]:
new = NN_Classic(**config_classic)
safetensors.torch.load_model(new, path_classic_directory + '/model.safetensors')
print(all(torch.equal(a, b) for a, b in zip(model_classic.state_dict().values(), new.state_dict().values())))

True


In [16]:
new = NN_Thinking(**config_thinking)
safetensors.torch.load_model(new, path_thinking_directory + '/model.safetensors')
print(all(torch.equal(a, b) for a, b in zip(model_thinking.state_dict().values(), new.state_dict().values())))

True


In [17]:
p = [
    '/kaggle/input/datasets/absolute1000/mini-llm-sprint4-ref/decoder.onnx',
    '/kaggle/input/datasets/absolute1000/mini-llm-sprint4-ref/encoder.onnx',
]
for _ in p:
    shutil.copy(_, path_classic_directory + '/onnx/')

In [18]:
shutil.copy('/kaggle/input/datasets/absolute1000/mini-llm-sprint4-ref/config.json', path_classic_directory)
shutil.copy('/kaggle/input/datasets/absolute1000/mini-llm-sprint4-ref/tokenizer.json', path_classic_directory)

'/kaggle/working/hf/classic/tokenizer.json'

In [19]:
PAD_ID, MASK_ID, USER_ID, BOT_ID, EOS_ID = (0, 1, 2, 3, 4)
SAMPLER_SFT_THINKING = {
    "mean_length_scale": 1.0,
    "min_len": 3,
    "n_candidates": 4,
    "block_len": 8,
    "repeat_penalty": 1.05,
    "ngram_ans_len": 3,
    "ngram_user_len": 4,
    "ngram_penalty": 5.0,
    "temperature": 0.5,
    "top_k": 50,
    "margin_for_gstar_replace": 0.3,
    "ANSWER_LEN": 64,
    "block_rounds": 2,
    "global_rounds": 2,
    'greedy': False
}
MODEL_INFERENCE_CONFIG = {
    'PAD_ID': PAD_ID,
    'MASK_ID': MASK_ID,
    'USER_ID' : USER_ID,
    'BOT_ID': BOT_ID,
    'EOS_ID': EOS_ID,
    'CONTEXT_LEN': 256,
    'ANSWER_LEN': 64,
    'PAD_SIDE': "left",
    'SAMPLER_SETTINGS': SAMPLER_SFT_THINKING,
    'MODEL_SETTINGS': config_thinking
}

In [20]:
shutil.copy('/kaggle/input/datasets/absolute1000/mini-llm-sprint4-ref/tokenizer.json', path_thinking_directory)
with open(path_thinking_directory + '/config.json', 'w', encoding='UTF-8') as f:
    json.dump(MODEL_INFERENCE_CONFIG, f)

In [21]:
%%writefile /kaggle/working/hf/space/sampler.py

import torch
import onnxruntime

PAD_ID, MASK_ID, USER_ID, BOT_ID, EOS_ID = 0, 1, 2, 3, 4

def ngram(seq, n):
    nxt, prv = dict(), dict()

    for i in range(0, len(seq) - n + 1):
        phrase_to_right = seq[i:i + n - 1]
        phrase_to_left = seq[i + 1:i + n]
        to_right_target = seq[i + n - 1]
        to_left_target = seq[i]
        if not(PAD_ID in phrase_to_right or PAD_ID == to_right_target):
            nxt.setdefault(tuple(phrase_to_right), set()).add(to_right_target)
        if not(PAD_ID in phrase_to_left or PAD_ID == to_left_target):
            prv.setdefault(tuple(phrase_to_left), set()).add(to_left_target)
    
    return nxt, prv

def penalize_ngram(ngram_end, ngram_start, ngram_len, ans_candidate_final, candidate_eos_idx, ans_candidate_logits, ngram_penalty, ANSWER_LEN):

    for i in range(candidate_eos_idx - ngram_len + 1):
        substring_for_ngram_to_end = ans_candidate_final[0, i:i + ngram_len - 1].tolist()
        substring_for_ngram_to_start = ans_candidate_final[0, i + 1:i + ngram_len].tolist()

        end_cands_to_penalize = ngram_end.get(tuple(substring_for_ngram_to_end), None)
        start_cands_to_penalize = ngram_start.get(tuple(substring_for_ngram_to_start), None)

        if end_cands_to_penalize is not None and MASK_ID not in substring_for_ngram_to_end:
            for word in end_cands_to_penalize:
                ans_candidate_logits[:, i + ngram_len - 1, word] -= ngram_penalty

        if start_cands_to_penalize is not None and MASK_ID not in substring_for_ngram_to_start:
            for word in start_cands_to_penalize:
                ans_candidate_logits[:, i, word] -= ngram_penalty

def gstar(device, model, margin_for_gstar_replace, start_block_idx, end_block_idx, KV_cache, KV_mask,
          ans_candidate_final, context_ids, rounds, is_global=False):
    block_size = end_block_idx - start_block_idx
    indexer = torch.arange(start_block_idx, end_block_idx, device=device)  # [k] — позиции в ответе
    rows = torch.arange(block_size, device=device)                         # [k] — номера копий

    # last user phrase between last <user> and last <bot>
    context = context_ids[0].tolist()
    last_user = len(context) - 1 - context[::-1].index(USER_ID) if USER_ID in context else -1
    last_bot = len(context) - 1 - context[::-1].index(BOT_ID)
    last_user_phrase = set(context[last_user + 1:last_bot])

    def leave_one_out_probs():
        batch = ans_candidate_final.repeat(block_size, 1)            # [k, 64] — k answer copies
        batch[rows, indexer] = MASK_ID                               # pos indexer[r] masked in r_i copy
        temp_logits = model.decode(batch, KV_cache.expand(block_size, -1, -1),
                                   KV_mask.expand(block_size, -1, -1)).float()   # [k, 64, V]
        temp_logits = temp_logits[rows, indexer]                     # [k, V] only masked pos
        temp_logits[:, (MASK_ID, PAD_ID, EOS_ID)] = -float('inf')
        return torch.softmax(temp_logits, dim=-1)

    for step in range(rounds):
        temp_probs = leave_one_out_probs()
        cur_token = ans_candidate_final[0, indexer]                  # [k]
        p_cur = temp_probs[rows, cur_token]                          # [k]
        p_best, best_token = temp_probs.max(dim=-1)                  # [k], [k]

        token_to_replace, pr, diff = None, None, -float('inf')
        for pretender in range(block_size):
            is_bad = p_cur[pretender] < margin_for_gstar_replace * p_best[pretender]
            is_other = best_token[pretender] != cur_token[pretender]
            is_free = cur_token[pretender].item() not in last_user_phrase
            gap = (p_best[pretender] - p_cur[pretender]).item()
            if is_bad and is_other and is_free and gap > diff:
                token_to_replace, pr, diff = best_token[pretender], pretender, gap

        if token_to_replace is None:
            break
        ans_candidate_final[0, indexer[pr]] = token_to_replace

    if is_global:
        temp_probs = leave_one_out_probs()
        p_cur = temp_probs[rows, ans_candidate_final[0, indexer]]
        return p_cur.clamp_min(1e-9).log().mean().item()

@torch.no_grad()
def generate(model, 
             context_ids, 
             mean_length_scale, 
             min_len, 
             n_candidates, 
             block_len, 
             repeat_penalty, 
             ngram_ans_len, 
             ngram_user_len, 
             ngram_penalty, 
             temperature,
             top_k,
             margin_for_gstar_replace,
             ANSWER_LEN,
             block_rounds,
             global_rounds,
             greedy,
             device
            ):
    context_ids = context_ids[None, :] # [256] -> [1, 256]
    KV_cache, KV_mask = model.encode(context_ids) # [1, 256, 384], [1, 1, 384]
    #answer_logits = torch.ones(size=(1, 64), dtype=torch.long) [1, 64]
    answer_probs = torch.nn.functional.softmax(
        model.decode(torch.full((1, ANSWER_LEN), MASK_ID, dtype=torch.long, device=device), KV_cache, KV_mask).float(), dim=2) # [1, 64, 12288]
    
    Len_mean = ((1 - answer_probs[..., EOS_ID] - answer_probs[..., PAD_ID]).sum(dim=1) * mean_length_scale).item()
    length_scale_candidates = torch.linspace(torch.tensor(0.8), torch.tensor(1.2), n_candidates) if n_candidates > 1 else torch.tensor([1.0])
    Len_candidates = ((Len_mean * length_scale_candidates).round()).clamp(min_len, 63).long() # [n_candidates]

    ngram_end_user, ngram_start_user = ngram(context_ids[0].tolist(), ngram_user_len)

    best_candidate, best_candidate_score = None, -float('inf')
    for idx in range(n_candidates):
        ans_candidate_final = torch.full((1, ANSWER_LEN), MASK_ID, dtype=torch.long, device=device)
        ans_candidate_final[:, Len_candidates[idx]] = EOS_ID
        ans_candidate_final[:, Len_candidates[idx] + 1:] = PAD_ID

        for start_block_idx in range(0, Len_candidates[idx].item(), block_len):
            end_block_idx = min(start_block_idx + block_len, Len_candidates[idx].item())

            block_region = torch.zeros(size=(64,), device=device).bool()
            block_region[start_block_idx:end_block_idx] = True

            while True:
                masks_still_to_open = (ans_candidate_final[0] == MASK_ID) & block_region
                if not masks_still_to_open.any().item(): break

                ans_candidate_logits = model.decode(ans_candidate_final, KV_cache, KV_mask).float()

                #penalty wrong end
                ans_candidate_logits[..., (MASK_ID, PAD_ID, EOS_ID)] = -float('inf')

                #penalty repeat
                used = ans_candidate_final[0][ans_candidate_final[0] != MASK_ID].unique()
                lu = ans_candidate_logits[0, :, used]
                ans_candidate_logits[0, :, used] = torch.where(lu > 0, lu / repeat_penalty, lu * repeat_penalty)

                #penalty ngram
                ngrams_ans_end, ngrams_ans_start = ngram(ans_candidate_final[0].tolist(), ngram_ans_len)
                penalize_ngram(ngrams_ans_end, ngrams_ans_start, ngram_ans_len, ans_candidate_final, Len_candidates[idx].item(), 
                               ans_candidate_logits, ngram_penalty, ANSWER_LEN)
                penalize_ngram(ngram_end_user, ngram_start_user, ngram_user_len, ans_candidate_final, Len_candidates[idx].item(), 
                               ans_candidate_logits, ngram_penalty, ANSWER_LEN)

                # temperature
                ans_candidate_logits[0] /= temperature

                #top k
                min_logit_per_row = ans_candidate_logits.topk(top_k, dim=2).values[:, :, -1:] # [64, 1]
                ans_candidate_logits = ans_candidate_logits.masked_fill(ans_candidate_logits < min_logit_per_row, -float("inf"))

                # probs and word choosing
                ans_candidate_probs = torch.softmax(ans_candidate_logits, 2).squeeze(0) # [1, 64, 12288] -> [64, 12288]

                if greedy:
                    tokens_choice = torch.argmax(ans_candidate_probs, dim=1, keepdim=True)
                else:
                    tokens_choice = torch.multinomial(ans_candidate_probs, 1) # [64, 1]
                confidence = ans_candidate_probs.gather(-1, tokens_choice).squeeze(1) # [64, 1] -> [64]

                # open only one mask per step
                i = confidence.masked_fill(~masks_still_to_open, -1).argmax().item()
                ans_candidate_final[0, i] = tokens_choice[i, 0]

            # gstar
            gstar(device, model, margin_for_gstar_replace, start_block_idx, end_block_idx, KV_cache, KV_mask, ans_candidate_final, context_ids, 
                  rounds=block_rounds)
        this_candidate_score = gstar(device, model, margin_for_gstar_replace, 0, Len_candidates[idx].item(), KV_cache, KV_mask, ans_candidate_final, 
                                     context_ids, rounds=global_rounds, is_global=True)
        if this_candidate_score > best_candidate_score:
            best_candidate_score = this_candidate_score
            best_candidate = ans_candidate_final
            
    return best_candidate.squeeze(0)

class OrtBackend():

    def __init__(self, enc_path, dec_path):
        self.encoder = onnxruntime.InferenceSession(enc_path, providers=['CPUExecutionProvider'])
        self.decoder = onnxruntime.InferenceSession(dec_path, providers=['CPUExecutionProvider'])

    def encode(self, context_ids):
        # context_ids [1, 256] torch.tensor
        memory, mask = self.encoder.run(None, {'context_ids': context_ids.numpy()})
        return torch.from_numpy(memory), torch.from_numpy(mask)

    def decode(self, answer_ids, memory, mask):
        # 
        return torch.from_numpy(self.decoder.run(None, {'memory': memory[:1].numpy(), 'mask': mask[:1].numpy(), 'answer_ids': answer_ids.numpy()})[0])

Writing /kaggle/working/hf/space/sampler.py


In [22]:
shutil.copy(f'{path_space_directory}/sampler.py', path_classic_directory)
shutil.copy(f'{path_space_directory}/sampler.py', path_thinking_directory)

'/kaggle/working/hf/thinking/sampler.py'

In [23]:
%%writefile {path_space_directory}/app.py
import json
import sys

import gradio as gr
import safetensors.torch
import torch
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

from sampler import generate, OrtBackend, PAD_ID, USER_ID, BOT_ID, EOS_ID

CLASSIC_REPO = 'quark28/mini-llm-classic-sft'
THINKING_REPO = 'quark28/mini-llm-thinking-sft'
CONTEXT_LEN = 256

torch.set_num_threads(2)  # free Space = 2 CPU cores


def load_config(folder):
    with open(f'{folder}/config.json', encoding='utf-8') as f:
        return json.load(f)


# ---------- loading both models once, at Space start ----------
classic_dir = snapshot_download(CLASSIC_REPO)    # local folder with all files of the repo
thinking_dir = snapshot_download(THINKING_REPO)
classic_cfg = load_config(classic_dir)
thinking_cfg = load_config(thinking_dir)

# classic: ONNX graphs through onnxruntime
classic_model = OrtBackend(f'{classic_dir}/onnx/encoder.onnx', f'{classic_dir}/onnx/decoder.onnx')

# thinking: plain PyTorch, class code comes from the repo itself
sys.path.insert(0, thinking_dir)
from modeling_thinking import NN_Thinking

thinking_model = NN_Thinking(**thinking_cfg['MODEL_SETTINGS'])
safetensors.torch.load_model(thinking_model, f'{thinking_dir}/model.safetensors')
thinking_model.eval()

MODELS = {'classic': classic_model, 'thinking': thinking_model}
SETTINGS = {
    'classic': {**classic_cfg['SAMPLER_SETTINGS'], 'greedy': False},
    'thinking': {**thinking_cfg['SAMPLER_SETTINGS'], 'greedy': False},
}
tok = Tokenizer.from_file(f'{classic_dir}/tokenizer.json')  # same tokenizer for both models


# ---------- chat history -> model input ----------
def as_text(content):
    # gradio passes text either as a str or as a list of parts [{'type': 'text', 'text': ...}]
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get('text', '')
    if isinstance(content, (list, tuple)):
        return ''.join(as_text(part) for part in content)
    return ''


def trim_history(ids, context_len=CONTEXT_LEN):
    # cut from the left on a user-turn boundary, as in Sprint 2
    if len(ids) <= context_len:
        return ids
    for i, t in enumerate(ids):
        if t == USER_ID and len(ids) - i <= context_len:
            return ids[i:]
    return ids[-context_len:]


def build_context(message, history):
    # same template as SFT: <user> text <bot> reply <eos> ... <user> message <bot>
    ids = []
    for turn in history:
        text_ids = tok.encode(as_text(turn['content'])).ids
        if turn['role'] == 'user':
            ids += [USER_ID] + text_ids + [BOT_ID]
        else:
            ids += text_ids + [EOS_ID]
    ids += [USER_ID] + tok.encode(as_text(message)).ids + [BOT_ID]
    ids = trim_history(ids)
    return [PAD_ID] * (CONTEXT_LEN - len(ids)) + ids  # left padding, as in training


def respond(message, history, model_name):
    context = torch.tensor(build_context(message, history), dtype=torch.long)
    answer = generate(MODELS[model_name], context_ids=context, device='cpu', **SETTINGS[model_name]).tolist()
    return tok.decode(answer[:answer.index(EOS_ID)])


demo = gr.ChatInterface(
    respond,
    additional_inputs=[gr.Dropdown(['classic', 'thinking'], value='classic', label='Model')],
    additional_inputs_accordion=gr.Accordion(label='Model', open=True),
    title='mini-llm',
    description=(
        'A 20M-parameter masked-diffusion chat model trained from scratch. English only. '
        'Answers are short and often wrong — it is a small research model, not an assistant. '
        '[GitHub](https://github.com/quark28/mini-llm)'
    ),
    examples=[
        ['Hello! How are you today?', 'classic'],
        ['I just got a new puppy!', 'classic'],
        ['The weather is terrible today, it is raining all day.', 'classic'],
        ['Do you have any pets?', 'thinking'],
    ],
)

if __name__ == '__main__':
    demo.launch()

Writing /kaggle/working/hf/space/app.py


In [24]:
%%writefile {path_space_directory}/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch
onnxruntime
tokenizers
safetensors

Writing /kaggle/working/hf/space/requirements.txt


In [25]:
%%writefile {path_space_directory}/README.md
---
title: mini-llm chat
emoji: 💬
colorFrom: indigo
colorTo: gray
sdk: gradio
sdk_version: 6.30.0
app_file: app.py
models:
  - quark28/mini-llm-classic-sft
  - quark28/mini-llm-thinking-sft
license: cc-by-nc-sa-4.0
short_description: 20M masked-diffusion chat model trained from scratch
---


Writing /kaggle/working/hf/space/README.md


In [26]:
from huggingface_hub import ModelCard, ModelCardData, EvalResult

GITHUB = 'https://github.com/quark28/mini-llm'
CLASSIC_REPO = f'{user}/mini-llm-classic-sft'
THINKING_REPO = f'{user}/mini-llm-thinking-sft'
SPACE_REPO = f'{user}/mini-llm-chat'

# key in the score dicts -> (HF dataset id, config, display name, metric)
BENCH = {
    'PIQA':       ('ybisk/piqa', None, 'PIQA', 'acc'),
    'HellaSwag*': ('Rowan/hellaswag', None, 'HellaSwag', 'acc_norm'),
    'ARC-e*':     ('allenai/ai2_arc', 'ARC-Easy', 'ARC-Easy', 'acc_norm'),
    'ARC-c*':     ('allenai/ai2_arc', 'ARC-Challenge', 'ARC-Challenge', 'acc_norm'),
    'WinoG':      ('allenai/winogrande', 'winogrande_xl', 'WinoGrande', 'acc'),
    'SciQ':       ('allenai/sciq', None, 'SciQ', 'acc'),
    'LAMBADA':    ('EleutherAI/lambada_openai', None, 'LAMBADA (OpenAI)', 'acc'),
    'BLiMP':      ('nyu-mll/blimp', None, 'BLiMP', 'acc'),
}
random_baseline = {'PIQA': 50.0, 'HellaSwag*': 25.0, 'ARC-e*': 25.0, 'ARC-c*': 25.0,
                   'WinoG': 50.0, 'SciQ': 25.0, 'LAMBADA': 0.0, 'BLiMP': 50.0}

TRAIN_DATASETS = ['HuggingFaceFW/fineweb-edu', 'li2017dailydialog/daily_dialog', 'facebook/empathetic_dialogues',
                  'bavard/personachat_truecased', 'google-research-datasets/mbpp']


def card_data(model_name, scores, tags):
    results = [EvalResult(task_type='text-generation', dataset_type=hf_id, dataset_name=name, dataset_config=cfg,
                          metric_type=metric, metric_value=scores[key], metric_name=f'{metric} (0-shot)')
               for key, (hf_id, cfg, name, metric) in BENCH.items()]
    return ModelCardData(model_name=model_name, language='en', license='cc-by-nc-sa-4.0',
                         pipeline_tag='text-generation', tags=tags, datasets=TRAIN_DATASETS, eval_results=results)


def bench_table(rows):
    head = '| Model | Params | ' + ' | '.join(BENCH[k][2] + ('*' if k.endswith('*') else '') for k in BENCH) + ' |'
    sep = '|---|---|' + '---|' * len(BENCH)
    lines = [head, sep]
    for name, params, scores in rows:
        lines.append(f'| {name} | {params} | ' + ' | '.join(f'{scores[k]:.1f}' for k in BENCH) + ' |')
    return '\n'.join(lines)


n_classic = sum(p.numel() for p in model_classic.parameters()) / 1e6
n_thinking = sum(p.numel() for p in model_thinking.parameters()) / 1e6

table = bench_table([
    ('**mini-llm classic SFT**', f'{n_classic:.1f}M', classic_sft),
    ('mini-llm Thinking SFT', f'{n_thinking:.1f}M', thinking_sft),
    ('Pythia-14m', '14M', pythia_14m),
    ('GPT-2', '124M', gpt2_124m),
    ('random', '—', random_baseline),
])

BENCH_NOTES = f"""Zero-shot, [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) 0.4.13, accuracy in %.
`*` — length-normalized accuracy (`acc_norm`). Our models are scored with an exact diffusion likelihood:
the answer slot is unmasked left to right and the log-probability of each revealed token is summed.
Answer options longer than the 64-token slot are truncated (PIQA 8.5% of options, HellaSwag 1.7%, others 0%).

These are base-level language benchmarks: they measure likelihood on raw text, not dialogue.
At 20M parameters both models are close to chance on reasoning tasks (ARC, WinoGrande, HellaSwag)."""

SAMPLER_TEXT = """Generation is not left-to-right. For every reply:

1. **Length.** One decoder pass over a fully masked slot; the expected reply length is
   `Σ (1 − p(<eos>) − p(<pad>))` over positions. Four candidates are generated with lengths ×0.8 … ×1.2.
2. **Block-wise unmasking.** The slot is filled in blocks of 8 tokens; inside a block, one masked
   position with the highest confidence is revealed per step (temperature 0.5, top-k 50,
   repetition penalty, n-gram bans against the reply and the user's last message).
3. **Remasking.** After each block and at the end, every token is re-scored with itself masked
   (leave-one-out); a token whose probability is below `margin × p(best alternative)` is replaced.
   This is a training-free variant of G-Star remasking ([arXiv:2510.08369](https://arxiv.org/abs/2510.08369)).
4. **Selection.** The candidate with the highest mean leave-one-out log-probability is returned.

The full sampler is in `sampler.py`; its settings are in `config.json` → `SAMPLER_SETTINGS`."""

classic_body = f"""
# mini-llm · classic SFT

A **20M-parameter chat model trained from scratch** with a **masked-diffusion** objective:
an encoder reads the conversation, and a small decoder fills a 64-token answer slot by iterative unmasking
instead of writing left to right. English small talk only.

**Try it:** [{SPACE_REPO}](https://huggingface.co/spaces/{SPACE_REPO}) · **Code:** [{GITHUB}]({GITHUB})

## Model

| | |
|---|---|
| Parameters | {n_classic:.1f}M (input and output embeddings tied) |
| Tokenizer | byte-level BPE, 12,288 tokens |
| Layout | encoder 6 blocks → decoder 2 blocks, d = 384, 6 heads |
| Blocks | pre-norm RMSNorm, SwiGLU FFN (1024), RoPE in self-attention; decoder cross-attends to the encoder |
| Input | context 256 tokens (left-padded) + answer slot 64 tokens |
| Objective | masked (absorbing-state) diffusion on the answer slot |

## Training

- **Pretraining** — FineWeb-Edu (`sample-10BT`).
- **SFT** — DailyDialog, EmpatheticDialogues, PersonaChat, MBPP and ~1.3k synthetic small-talk / polite-refusal dialogues.
  Prompt template: `<user> text <bot> reply <eos>`. The loss covers the whole answer slot including padding,
  so the model learns where to stop.

Trained on Kaggle T4 GPUs.

## Sampling

{SAMPLER_TEXT}

## Usage

ONNX (no PyTorch model code needed, only the sampler, which uses `torch` for tensors):

```python
import json, sys, torch
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

d = snapshot_download('{CLASSIC_REPO}')
sys.path.insert(0, d)
from sampler import generate, OrtBackend, PAD_ID, USER_ID, BOT_ID, EOS_ID

tok = Tokenizer.from_file(f'{{d}}/tokenizer.json')
cfg = json.load(open(f'{{d}}/config.json'))
model = OrtBackend(f'{{d}}/onnx/encoder.onnx', f'{{d}}/onnx/decoder.onnx')

ids = [USER_ID] + tok.encode('Hello! How are you?').ids + [BOT_ID]
ids = [PAD_ID] * (256 - len(ids)) + ids
out = generate(model, torch.tensor(ids), device='cpu', **cfg['SAMPLER_SETTINGS']).tolist()
print(tok.decode(out[:out.index(EOS_ID)]))
```

PyTorch: `modeling_classic.py` + `model.safetensors`
(`safetensors.torch.load_model(NN_Classic(**cfg['MODEL_SETTINGS']), 'model.safetensors')`); the same `generate` works on it.

The ONNX graphs: `encoder.onnx` (`context_ids [1, 256]` → `memory`, `mask`) runs once per reply;
`decoder.onnx` (`memory`, `mask`, `answer_ids [batch, 64]` → `logits [batch, 64, 12288]`) runs on every step,
batch is dynamic for the remasking pass.

## Benchmarks

{table}

{BENCH_NOTES}

## Limitations

- English only; replies are short (usually one sentence).
- No real world knowledge; answers questions in the right *form* but not with the right *content*.
- Often misreads the emotion of a message, e.g. *"I lost my job yesterday."* → *"Oh no! I am glad you lost your job."*
- Not instruction-following, not safe for any real use. It is a learning / research project.

## License

Weights: CC BY-NC-SA 4.0 (the strictest license among the SFT datasets: DailyDialog is CC BY-NC-SA 4.0,
EmpatheticDialogues is CC BY-NC 4.0). Code: see the [GitHub repository]({GITHUB}).
"""

thinking_body = f"""
# mini-llm · Thinking SFT (research variant)

The same masked-diffusion encoder–decoder as
[{CLASSIC_REPO}](https://huggingface.co/{CLASSIC_REPO}), with an experimental **Thinking** module
between the encoder and the decoder. Kept for research; **for chat use the classic model.**

**Try it:** [{SPACE_REPO}](https://huggingface.co/spaces/{SPACE_REPO}) (select *thinking*) · **Code:** [{GITHUB}]({GITHUB})

## Model

| | |
|---|---|
| Parameters | {n_thinking:.1f}M (embeddings tied with the vocabulary projections) |
| Tokenizer | byte-level BPE, 12,288 tokens |
| Layout | encoder 4 blocks → Thinking → decoder 2 blocks, d = 384, 6 heads |
| Input | context 256 tokens (left-padded) + answer slot 64 tokens |
| Objective | masked (absorbing-state) diffusion on the answer slot |

**Thinking** projects encoder states onto the (tied) vocabulary embeddings and replaces the encoder output
with a per-position channel plus one context-level vector. Under the same training pipeline it does not beat
the plain encoder: SciQ −15.0 points, BLiMP +2.9, other differences are within noise.

Training data and procedure are the same as for the classic model (FineWeb-Edu pretraining → dialogue SFT).

## Usage

PyTorch only (no ONNX export for this variant):

```python
import json, sys, torch, safetensors.torch
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

d = snapshot_download('{THINKING_REPO}')
sys.path.insert(0, d)
from modeling_thinking import NN_Thinking
from sampler import generate, PAD_ID, USER_ID, BOT_ID, EOS_ID

tok = Tokenizer.from_file(f'{{d}}/tokenizer.json')
cfg = json.load(open(f'{{d}}/config.json'))
model = NN_Thinking(**cfg['MODEL_SETTINGS'])
safetensors.torch.load_model(model, f'{{d}}/model.safetensors')
model.eval()

ids = [USER_ID] + tok.encode('Hello! How are you?').ids + [BOT_ID]
ids = [PAD_ID] * (256 - len(ids)) + ids
out = generate(model, torch.tensor(ids), device='cpu', **cfg['SAMPLER_SETTINGS']).tolist()
print(tok.decode(out[:out.index(EOS_ID)]))
```

Sampling is the same as in the classic model card, with `margin = 0.3`.

## Benchmarks

{table}

{BENCH_NOTES}

## Limitations

Same as the classic model; in addition, replies drift off-topic and repeat words more often.

## License

Weights: CC BY-NC-SA 4.0 (see the classic model card). Code: see the [GitHub repository]({GITHUB}).
"""

ModelCard(f'---\n{card_data("mini-llm-classic-sft", classic_sft, ["masked-diffusion", "diffusion-language-model", "encoder-decoder", "from-scratch", "onnx"]).to_yaml()}\n---\n{classic_body}') \
    .save(f'{path_classic_directory}/README.md')
ModelCard(f'---\n{card_data("mini-llm-thinking-sft", thinking_sft, ["masked-diffusion", "diffusion-language-model", "encoder-decoder", "from-scratch"]).to_yaml()}\n---\n{thinking_body}') \
    .save(f'{path_thinking_directory}/README.md')
print(open(f'{path_classic_directory}/README.md').read()[:1500])


---
datasets:
- HuggingFaceFW/fineweb-edu
- li2017dailydialog/daily_dialog
- facebook/empathetic_dialogues
- bavard/personachat_truecased
- google-research-datasets/mbpp
language: en
license: cc-by-nc-sa-4.0
pipeline_tag: text-generation
tags:
- masked-diffusion
- diffusion-language-model
- encoder-decoder
- from-scratch
- onnx
model-index:
- name: mini-llm-classic-sft
  results:
  - task:
      type: text-generation
    dataset:
      name: PIQA
      type: ybisk/piqa
    metrics:
    - type: acc
      value: 54.5
      name: acc (0-shot)
  - task:
      type: text-generation
    dataset:
      name: HellaSwag
      type: Rowan/hellaswag
    metrics:
    - type: acc_norm
      value: 26.7
      name: acc_norm (0-shot)
  - task:
      type: text-generation
    dataset:
      name: ARC-Easy
      type: allenai/ai2_arc
      config: ARC-Easy
    metrics:
    - type: acc_norm
      value: 31.9
      name: acc_norm (0-shot)
  - task:
      type: text-generation
    dataset:
      name: ARC

In [27]:
api = huggingface_hub.HfApi()
for repo, folder in [(f'{user}/mini-llm-classic-sft', path_classic_directory),
                     (f'{user}/mini-llm-thinking-sft', path_thinking_directory)]:
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path=folder, repo_id=repo, ignore_patterns=['__pycache__/*', '*.pyc'])
    print('uploaded', repo, api.list_repo_files(repo))

uploaded quark28/mini-llm-classic-sft ['.gitattributes', 'README.md', 'config.json', 'model.safetensors', 'modeling_classic.py', 'onnx/decoder.onnx', 'onnx/encoder.onnx', 'sampler.py', 'tokenizer.json']
uploaded quark28/mini-llm-thinking-sft ['.gitattributes', 'README.md', 'config.json', 'model.safetensors', 'modeling_thinking.py', 'sampler.py', 'tokenizer.json']


In [29]:
!pip install -q gradio==6.30.0 onnxruntime
sys.path.insert(0, path_space_directory)
import app
for name in ['classic', 'thinking']:
    print(name, '->', app.respond('Hello! How are you today?', [], name))
hist = [{'role': 'user', 'content': 'I just got a new puppy!'}, {'role': 'assistant', 'content': 'What kind of puppy?'}]
print('dialog ->', app.respond('A golden retriever', hist, 'classic'))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

classic -> I am doing well. How are you?
thinking -> I am good. How are you? I'm doing well, how are you doing today?
dialog -> What kind of puppy?


In [30]:
space = f'{user}/mini-llm-chat'
api.create_repo(space, repo_type='space', space_sdk='gradio', exist_ok=True)
api.upload_folder(folder_path=path_space_directory, repo_id=space, repo_type='space',
                  ignore_patterns=['__pycache__/*', '*.pyc'])
print(f'https://huggingface.co/spaces/{space}')

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6ae0e307-3d26029a0e5f2d47347cc922;733f2afa-9579-45cd-b5ac-67ab5d929711)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro